## 16. Transform Orders Data - String to JSON — objective

- Compare raw fields with typed records and parsing failures.
- Distinguish query results from materialized table state.

Historical results; no code executed.
- [Original inputs: gizmobox/orders](../../../data/gizmobox/orders/) — 4 files.

## Learning notes — corrective commentary (originals unchanged)

Supplied orders JSONL includes malformed lines; inspect corrupt-record handling before interpreting row counts. Original files are linked above.

# 16. Transform Orders Data - String to JSON

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea03-etl-with-apache-spark/16.%20Transform%20Orders%20Data%20-%20String%20to%20JSON.json)

## Transform Orders Data - String to JSON
1. Pre-process the JSON String to fix the Data Quality Issues
1. Transform JSON String to JSON Object
1. Write transformed data to the silver schema

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * 
  FROM gizmobox.bronze.v_orders;
```

In [ ]:
# Archived output only; no computation.

value
"{""order_id"": 1, ""customer_id"": 6973, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 10:13:59"", ""total_amount"": 499, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 8, ""name""… [truncated]"
"{""order_id"": 2, ""customer_id"": 3532, ""order_date"": 2025-01-19, ""transaction_timestamp"": ""2025-01-19 00:05:13"", ""total_amount"": 985, ""payment_method"": ""PayPal"", ""items"": [{""item_id"": 4, ""name"": ""Smartw… [truncated]"
"{""order_id"": 3, ""customer_id"": 3532, ""order_date"": ""2025-01-08"", ""transaction_timestamp"": ""2025-01-08 23:11:00"", ""total_amount"": 597, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 3, ""name""… [truncated]"
"{""order_id"": 4, ""customer_id"": 3532, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 05:49:26"", ""total_amount"": 999, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 2, ""name""… [truncated]"
"{""order_id"": 8, ""customer_id"": 9179, ""order_date"": ""2025-01-02"", ""transaction_timestamp"": ""2025-01-02 13:11:15"", ""total_amount"": 399, ""payment_method"": ""PayPal"", ""items"": [{""item_id"": 5, ""name"": ""Tabl… [truncated]"
"{""order_id"": 14, ""customer_id"": 7207, ""order_date"": ""2025-01-01"", ""transaction_timestamp"": ""2025-01-01 18:22:26"", ""total_amount"": 597, ""payment_method"": ""PayPal"", ""items"": [{""item_id"": 3, ""name"": ""Wir… [truncated]"
"{""order_id"": 18, ""customer_id"": 4468, ""order_date"": ""2025-01-22"", ""transaction_timestamp"": ""2025-01-22 21:12:08"", ""total_amount"": 799, ""payment_method"": ""Credit Card"", ""items"": [{""item_id"": 10, ""name""… [truncated]"
"{""order_id"": 22, ""customer_id"": 4761, ""order_date"": 2025-01-01, ""transaction_timestamp"": ""2025-01-01 15:49:19"", ""total_amount"": 199, ""payment_method"": ""PayPal"", ""items"": [{""item_id"": 3, ""name"": ""Wirel… [truncated]"
"{""order_id"": 26, ""customer_id"": 7007, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 19:01:25"", ""total_amount"": 699, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 1, ""name… [truncated]"
"{""order_id"": 29, ""customer_id"": 5953, ""order_date"": ""2025-01-18"", ""transaction_timestamp"": ""2025-01-18 03:05:04"", ""total_amount"": 129, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 7, ""name… [truncated]"


### 1. Pre-process the JSON String to fix the Data Quality Issues
[regexp_replace function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_replace)

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TEMP VIEW tv_orders_fixed
AS
SELECT value,
       regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_value 
  FROM gizmobox.bronze.v_orders;
```

In [ ]:
# Archived output only; no computation.

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

## Learning notes — before source operation

**json-array**: JSON extraction and array operations change structure: parsing creates typed values, while explode can create multiple rows per input. Inspect missing fields, null arrays and empty arrays.

### 2. Transform JSON String to JSON Object
- Function [schema_of_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/schema_of_json)
- Function [from_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/from_json)

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT schema_of_json(fixed_value) AS schema,
       fixed_value
  FROM tv_orders_fixed
 LIMIT 1;
```

In [ ]:
# Archived output only; no computation.

schema,fixed_value
"STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, o… [truncated]","{""order_id"": 1, ""customer_id"": 6973, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 10:13:59"", ""total_amount"": 499, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 8, ""name""… [truncated]"


Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT from_json(fixed_value, 
                 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value,
       fixed_value
  FROM tv_orders_fixed;
```

In [ ]:
# Archived output only; no computation.

json_value,fixed_value
[nested value omitted],"{""order_id"": 1, ""customer_id"": 6973, ""order_date"": ""2025-01-05"", ""transaction_timestamp"": ""2025-01-05 10:13:59"", ""total_amount"": 499, ""payment_method"": ""Bank Transfer"", ""items"": [{""item_id"": 8, ""name""… [truncated]"


### 3. Write transformed data to the silver schema

Archived Databricks source — shown for reading, not Python execution.

```sql
DROP TABLE IF EXISTS gizmobox.silver.orders_json;
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders_json
AS
SELECT from_json(fixed_value, 
                 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value
  FROM tv_orders_fixed;
```

In [ ]:
# Archived output only; no computation.

num_affected_rows,num_inserted_rows


Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM gizmobox.silver.orders_json;
```

In [ ]:
# Archived output only; no computation.

json_value
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
